# Análisis Exploratorio de Datos - Tráfico Vehicular Medellín 2019-2022

**Proyecto Integrador - Talento Tech**  
Análisis de datos de movilidad urbana para la optimización del tráfico en Medellín.

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 5)
print("Librerías cargadas correctamente.")

## 1. Carga de Datos

Cargamos el dataset unificado de tráfico (Parquet) y el de incidentes.

In [ ]:
# Cargar datos de tráfico
df = pd.read_parquet('data/trafico_medellin_unificado.parquet')
print(f"Registros de tráfico: {len(df):,}")
print(f"Columnas: {list(df.columns)}")

# Cargar incidentes
df_inc = pd.read_parquet('data/incidentes_limpios.parquet')
print(f"\nRegistros de incidentes: {len(df_inc):,}")
print(f"Columnas: {list(df_inc.columns)}")

## 2. Vista Preliminar de los Datos

In [ ]:
print("=== TRÁFICO ===")
print(df.head(3).to_string())
print("\n")
print(df.info())
print("\n")
print(df.describe())

In [ ]:
print("=== INCIDENTES ===")
print(df_inc.head(3).to_string())
print("\n")
print(df_inc.info())

## 3. Valores Nulos y Calidad de Datos

In [ ]:
# Nulos en tráfico
nulos = df.isnull().sum()
nulos_pct = (nulos / len(df)) * 100
pd.DataFrame({'nulos': nulos, 'porcentaje': nulos_pct}).sort_values('nulos', ascending=False).head(10)

In [ ]:
# Nulos en incidentes
nulos_inc = df_inc.isnull().sum()
pd.DataFrame({'nulos': nulos_inc, 'porcentaje': (nulos_inc/len(df_inc))*100})

## 4. Análisis Temporal

In [ ]:
# Evolución por año
anual = df.groupby('anio').agg(
    registros=('velocidad_kmh', 'count'),
    velocidad_prom=('velocidad_kmh', 'mean'),
    flujo_prom=('intensidad_veh_h', 'mean'),
    ocupacion_prom=('ocupacion', 'mean'),
    congestion=('indice_congestion', 'mean')
).reset_index()
anual['congestion'] = anual['congestion'] * 100
print(anual.round(1).to_string(index=False))

# Gráfico
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(anual['anio'], anual['velocidad_prom'], 'o-', color='green', linewidth=2)
axes[0].set_title('Velocidad Promedio por Año')
axes[0].set_xlabel('Año'); axes[0].set_ylabel('km/h')

axes[1].plot(anual['anio'], anual['congestion'], 'o-', color='red', linewidth=2)
axes[1].set_title('Índice de Congestión por Año')
axes[1].set_xlabel('Año'); axes[1].set_ylabel('%')
plt.tight_layout()
plt.show()

In [ ]:
# Distribución por hora
hora = df.groupby('hora').agg(
    velocidad=('velocidad_kmh', 'mean'),
    flujo=('intensidad_veh_h', 'mean')
).reset_index()

fig, ax1 = plt.subplots(figsize=(12, 4))
ax1.bar(hora['hora'], hora['flujo'], alpha=0.6, color='blue', label='Flujo (veh/h)')
ax2 = ax1.twinx()
ax2.plot(hora['hora'], hora['velocidad'], 'o-', color='green', linewidth=2, label='Velocidad (km/h)')
ax1.set_xlabel('Hora del día'); ax1.set_ylabel('Flujo (veh/h)')
ax2.set_ylabel('Velocidad (km/h)')
ax1.set_title('Velocidad y Flujo por Hora del Día')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
plt.show()

In [ ]:
# Día de semana
orden = ['Lunes', 'Martes', 'Miercoles', 'Jueves', 'Viernes', 'Sabado', 'Domingo']
df['dia_semana'] = pd.Categorical(df['dia_semana'], categories=orden, ordered=True)
semana = df.groupby('dia_semana', observed=True).agg(
    flujo=('intensidad_veh_h', 'mean'),
    velocidad=('velocidad_kmh', 'mean')
).reset_index()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(semana['dia_semana'], semana['flujo'], color='steelblue')
ax.set_title('Flujo Promedio por Día de la Semana')
ax.set_ylabel('Vehículos/h')
plt.show()

## 5. Análisis por Corredor

In [ ]:
# Top corredores por congestión
top_cong = df[~df['corredor'].isin(['', 'Nan', 'None'])].groupby('corredor').agg(
    velocidad=('velocidad_kmh', 'mean'),
    congestion=('indice_congestion', 'mean'),
    registros=('velocidad_kmh', 'count')
).reset_index()
top_cong['congestion'] = top_cong['congestion'] * 100
top_cong = top_cong[top_cong['registros'] > 10000].sort_values('congestion', ascending=False).head(10)

plt.figure(figsize=(12, 5))
plt.barh(top_cong['corredor'], top_cong['congestion'], color='coral')
plt.xlabel('% Congestión')
plt.title('Top 10 Corredores Más Congestionados')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Top por volumen
top_vol = df[~df['corredor'].isin(['', 'Nan', 'None'])].groupby('corredor').agg(
    volumen=('intensidad_veh_h', 'sum'),
    velocidad=('velocidad_kmh', 'mean')
).reset_index().sort_values('volumen', ascending=False).head(10)

plt.figure(figsize=(12, 5))
bars = plt.bar(top_vol['corredor'], top_vol['volumen'], color='steelblue')
plt.xticks(rotation=45, ha='right')
plt.ylabel('Volumen total (veh)')
plt.title('Top 10 Corredores por Volumen Vehicular')
plt.tight_layout()
plt.show()

## 6. Análisis de Incidentes

In [ ]:
# Incidentes por tipo y gravedad
print("\n--- Incidentes por Tipo ---")
print(df_inc['tipo_evento'].value_counts())
print("\n--- Incidentes por Gravedad ---")
print(df_inc['gravedad'].value_counts())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
df_inc['tipo_evento'].value_counts().plot(kind='bar', ax=axes[0], color='tomato')
axes[0].set_title('Incidentes por Tipo')
axes[0].set_xlabel(''); axes[0].tick_params(axis='x', rotation=45)

df_inc['gravedad'].value_counts().plot(kind='pie', ax=axes[1], autopct='%1.1f%%', colors=['red', 'orange', 'gold', 'lightgreen'])
axes[1].set_title('Distribución por Gravedad')
plt.tight_layout()
plt.show()

## 7. Conclusiones del EDA

1. **Tendencia negativa**: La velocidad promedio cayó de 27.8 km/h (2019) a 25.5 km/h (2022), mientras la congestión subió de 21.7% a 27.4%.
2. **2020 atípico**: Menor flujo vehicular por pandemia, lo que redujo artificialmente la congestión.
3. **Horas críticas**: Entre 6-9 am y 5-8 pm se concentra la mayor congestión con velocidades menores a 20 km/h.
4. **Corredores críticos**: Existen corredores específicos con índices de congestión superiores al 40% que requieren intervención prioritaria.
5. **Incidentes**: Accidentes y congestión son los eventos más frecuentes (~11,000 cada uno). El 50% de los incidentes son críticos o graves.
6. **Estacionalidad**: Los patrones mensuales se mantienen consistentes entre años, validando la calidad de los datos.

---

